In [2]:
import pandas as pd
import os

First we need to calculate the parameters required for the strategy. The following are the require parameters:
- Relative strength(RS) this is the relative stregth of the stock with respect to nifty
- Relative strength index for 10 periods
- EMA 20 , 50 , 100


In [3]:
nifty_df = pd.read_csv('data/OHLC/NIFTY.csv')
nifty_df["nifty_ratio_5"] = round(nifty_df["close"] / nifty_df["close"].shift(5),2)
nifty_df.to_csv('data/NIFTY.csv')

Rounding the OHLC and Volume

In [ ]:
symbols_df = pd.read_csv('symbols.csv')
for symbol in symbols_df.symbols:
    symbol_df = pd.read_csv(f'data/OHLC/{symbol}.csv')
    symbol_df.close=round(symbol_df.close,2)
    symbol_df.high=round(symbol_df.high,2)
    symbol_df.low=round(symbol_df.low,2)
    symbol_df.open=round(symbol_df.open,2)
    symbol_df.volume=round(symbol_df.volume)
    symbol_df.to_csv(f"data/OHLC/{symbol}.csv",index=False)

## Relative Strength (RS)

$$
RS = \frac{\left(\frac{N_t}{N_{t-5}}\right)}
{\left(\frac{S_t}{S_{t-5}}\right)} - 1
$$

Where:

$$
\begin{aligned}
RS      & = \text{Relative Strength} \\
N_t     & = \text{Comparative index close at time } t \\
N_{t-5} & = \text{Comparative index close 5 periods before } t \\
S_t     & = \text{Stock close at time } t \\
S_{t-5} & = \text{Stock close 5 periods before } t
\end{aligned}
$$


Here we are going to use NIFTY as the Comparative index

In [ ]:

for symbol in symbols_df.symbols:
    symbol_df = pd.read_csv(f'data/OHLC/{symbol}.csv')
    symbol_df['stock_ratio_5'] = round(symbol_df['close']/symbol_df['close'].shift(5),2)
    symbol_df['RS']=round((symbol_df['stock_ratio_5']/nifty_df['nifty_ratio_5']) -1,2)
    symbol_df.to_csv(f"data/OHLC/{symbol}.csv",index=False)


Calculating the RSI of period 10 for every stock

In [ ]:
def rsi(close,period=10):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/period,adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period,adjust=False).mean()

    rs = avg_gain/avg_loss

    return 100 - (100 / (1+rs))

for symbol in symbols_df.symbols:
    symbol_df = pd.read_csv(f'data/OHLC/{symbol}.csv')
    symbol_df['RSI_10'] = round(rsi(symbol_df.close,10),2)
    symbol_df.to_csv(f"data/OHLC/{symbol}.csv",index=False)

Calculating the ema 20,50,100 for all symbols

In [10]:
def ema(close, period):
    return close.ewm(span=period,adjust=False).mean()

for symbol in symbols_df.symbols:
    symbol_df = pd.read_csv(f'data/OHLC/{symbol}.csv')
    symbol_df['EMA_20'] = round(ema(symbol_df.close,20),2)
    symbol_df['EMA_50'] = round(ema(symbol_df.close,50),2)
    symbol_df['EMA_100'] = round(ema(symbol_df.close,100),2)
    symbol_df.to_csv(f"data/OHLC/{symbol}.csv",index=False)